##### HIGH LEVEL IMPLEMENTATION OF THE PAPER 'LLaMA: Open and Efficient Foundation Language Models (Touvron et al., 2023)' USING PYTORCH

here is the link to the paper: https://arxiv.org/pdf/2302.13971

things to implement
- sentencepiece BPE
- SWIGLU
- ROPE
- PRENORM

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import Counter, defaultdict
import torch.nn.functional as F
import re
import math
import time


In [ ]:
with open("input.txt", "r", encoding="utf-8") as f:
    training_texts = f.read().strip().split("\n\n")
    training_texts = training_texts[:3]


In [ ]:
sen1 = "hello how are you doing"
sen2 = "what's  going on  here"
sen3 = """multi line sentence here doing things \n
second things """

rev1 = sen1.replace(" ", " _")
rev1 = re.sub(r"^.", r"_\g<0>", rev1)

rev2 = sen2.replace(" ", "_")
rev3 = sen3.replace(" ", "_")
rev3 = re.sub(r"^.", r"_\g<0>", rev3)

result = " ".join("_" + word for word in sen3.split())


print(rev1)
print(rev2)
print(rev3)
print(result)

In [ ]:
class LLaMABPETokenizer:
    def __init__(self):
        # 1. Special Control Tokens
        self.special_tokens = ["<unk>", "<s>", "</s>"]
        self.word2idx = {tok: idx for idx, tok in enumerate(self.special_tokens)}

        # 2. Add 256 Raw Byte Tokens (<0x00> to <0xFF>) for Byte Fallback
        self.byte_tokens = [f"<0x{i:02X}>" for i in range(256)]
        for byte_tok in self.byte_tokens:
            self.word2idx[byte_tok] = len(self.word2idx)

        # 3. Add SentencePiece Space Character (' ', U+2581)
        self.SPIECE_UNDERLINE = "▅"
        if self.SPIECE_UNDERLINE not in self.word2idx:
            self.word2idx[self.SPIECE_UNDERLINE] = len(self.word2idx)

        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.ranks = {}  # Stores merge priority (lower index = higher priority)
        self.word_freqs = Counter()

    def _preprocess_text(self, text):
        """Replaces spaces with SentencePiece underline ' ' and prepends ' '."""
        text = text.replace(" ", self.SPIECE_UNDERLINE)
        if not text.startswith(self.SPIECE_UNDERLINE):
            text = self.SPIECE_UNDERLINE + text
        return text

    def preprocess_and_count(self, corpus):
        """Converts raw text corpus into character tuples and counts frequencies."""
        raw_counts = Counter()
        for sentence in corpus:
            processed = self._preprocess_text(sentence)
            raw_counts[processed] += 1

        self.word_freqs = Counter()
        for text, count in raw_counts.items():
            # Convert text into a tuple of unicode characters
            char_tuple = tuple(list(text))
            self.word_freqs[char_tuple] = count

        # Add initial base characters to vocabulary
        for word_tuple in self.word_freqs.keys():
            for char in word_tuple:
                if char not in self.word2idx:
                    idx = len(self.word2idx)
                    self.word2idx[char] = idx
                    self.idx2word[idx] = char

    def get_pair_counts(self):
        pairs = defaultdict(int)
        for word_tuple, freq in self.word_freqs.items():
            for i in range(len(word_tuple) - 1):
                pairs[(word_tuple[i], word_tuple[i + 1])] += freq
        return pairs

    def fit(self, corpus, num_merges):
        """Learns BPE merges over the input corpus."""
        self.preprocess_and_count(corpus)

        for rank in range(num_merges):
            pairs = self.get_pair_counts()
            if not pairs:
                break

            # Best pair is the highest frequency bigram
            best_pair = max(pairs, key=pairs.get)
            first, second = best_pair
            merged = first + second

            # Record rank priority
            self.ranks[best_pair] = rank

            if merged not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[merged] = idx
                self.idx2word[idx] = merged

            # In-place tuple updating
            new_word_freqs = Counter()
            for word_tuple, freq in self.word_freqs.items():
                new_tuple = []
                i = 0
                while i < len(word_tuple):
                    if (
                        i < len(word_tuple) - 1
                        and word_tuple[i] == first
                        and word_tuple[i + 1] == second
                    ):
                        new_tuple.append(merged)
                        i += 2
                    else:
                        new_tuple.append(word_tuple[i])
                        i += 1
                new_word_freqs[tuple(new_tuple)] = freq

            self.word_freqs = new_word_freqs

    def _tokenize_sequence(self, sequence):
        """Applies learned BPE ranks to a single character sequence."""
        sequence = list(sequence)

        while len(sequence) > 1:
            pairs = [(sequence[i], sequence[i + 1]) for i in range(len(sequence) - 1)]
            valid_pairs = [p for p in pairs if p in self.ranks]
            if not valid_pairs:
                break

            # Find pair learned earliest in training (highest priority)
            best_pair = min(valid_pairs, key=lambda p: self.ranks[p])
            first, second = best_pair

            new_sequence = []
            i = 0
            while i < len(sequence):
                if (
                    i < len(sequence) - 1
                    and sequence[i] == first
                    and sequence[i + 1] == second
                ):
                    new_sequence.append(first + second)
                    i += 2
                else:
                    new_sequence.append(sequence[i])
                    i += 1
            sequence = new_sequence

        return sequence

    def tokenize(self, text):
        """Tokenizes text using SentencePiece-style BPE with Byte Fallback."""
        processed = self._preprocess_text(text)
        tokens = self._tokenize_sequence(processed)

        final_tokens = []
        for token in tokens:
            if token in self.word2idx:
                final_tokens.append(token)
            else:
                # Byte Fallback for unseen UTF-8 characters
                for byte in token.encode("utf-8"):
                    final_tokens.append(f"<0x{byte:02X}>")

        return final_tokens

    def encode(self, text, add_bos=True):
        """Converts text into numerical token IDs (LLaMA style)."""
        tokens = self.tokenize(text)
        ids = [self.word2idx[t] for t in tokens]
        if add_bos:
            ids = [self.word2idx["<s>"]] + ids
        return ids

    def decode(self, ids):
        """Decodes token IDs back to human-readable string."""
        tokens = [
            self.idx2word[i] for i in ids if self.idx2word[i] not in ["<s>", "</s>"]
        ]

        byte_stream = bytearray()
        for tok in tokens:
            if tok.startswith("<0x") and tok.endswith(">"):
                # Handle Byte Fallback tokens
                byte_val = int(tok[4:-1], 16)
                byte_stream.append(byte_val)
            else:
                byte_stream.extend(tok.encode("utf-8"))

        text = byte_stream.decode("utf-8", errors="replace")
        text = text.replace(self.SPIECE_UNDERLINE, " ")
        return text.strip() if text.startswith(" ") else text

In [ ]:
class VocabularyBuilder:
    def __init__(self):
        # Special tokens required for translation models
        self.word2idx = {
            "<PAD>": 0,
            "<UNK>": 1,
            "<SOS>": 2,
            "<EOS>": 3,
            "</w>": 4,
            "<EXTRACT>": 5,
        }
        self.idx2word = {v: k for k, v in self.word2idx.items()}
        self.merge_order = []
        self.num_words = len(self.word2idx)

        # This will hold our word frequency dictionary
        self.word_freqs = Counter()

    def preprocess_and_count(self, all_sentences):
        """
        Processes sentences and builds a word frequency dictionary.
        This runs once and saves massive amounts of memory and time.
        """
        raw_word_counts = Counter()

        for sentence in all_sentences:
            sentence = sentence.lower()
            sentence = sentence.replace("\n", " ")
            sentence = sentence.replace(" ", "_")
            # sentence = re.sub(r"^[^:]+:\s*", "", sentence)
            sentence = re.sub(r'([.,!?;:"\'()])', r" \1 ", sentence)

            words = sentence.strip().split()
            raw_word_counts.update(words)

        # Convert words into spaces-separated characters with an end-of-word symbol
        self.word_freqs = Counter()
        for word, count in raw_word_counts.items():
            # Example: 'hug' -> 'h u g </w>'
            char_tuple = tuple(list(word) + ["</w>"])
            self.word_freqs[char_tuple] = count
        # Initialize basic single-character vocabulary
        self.build_initial_vocabulary()

    def build_initial_vocabulary(self):
        """Extracts unique base characters from our counted words."""
        for word_tuple in self.word_freqs.keys():
            for char in word_tuple:
                if char not in self.word2idx:
                    self.word2idx[char] = self.num_words
                    self.idx2word[self.num_words] = char
                    self.num_words += 1

    def get_pair_counts(self):
        """
        Fast pair counting using the pre-calculated word frequencies.
        """
        pairs = defaultdict(int)
        for word_tuple, freq in self.word_freqs.items():
            for i in range(len(word_tuple) - 1):
                pair = (word_tuple[i], word_tuple[i + 1])
                pairs[pair] += freq
        # print("pairs: ", pairs)
        return pairs

    def merge_vocab(self, pair):
        """
        Merges the most frequent pair in-place across the vocabulary.
        """
        new_word_freqs = Counter()
        bigram = re.escape(" ".join(pair))
        # Look for the pair with a boundary or space around it
        p = re.compile(r"(?<!\S)" + bigram + r"(?!\S)")
        # Save the merge order rule
        merged_string = "".join(pair)
        self.merge_order.append(pair)

        # Add the new merged token to the vocabulary if it isn't there
        if merged_string not in self.word2idx:
            self.word2idx[merged_string] = self.num_words
            self.idx2word[self.num_words] = merged_string
            self.num_words += 1

        # Update the character tuple configurations
        for word_tuple, freq in self.word_freqs.items():
            word_str = " ".join(word_tuple)
            new_word_str = p.sub(merged_string, word_str)
            new_word_tuple = tuple(new_word_str.split())
            new_word_freqs[new_word_tuple] = freq

        self.word_freqs = new_word_freqs

    def fit(self, all_sentences, num_merges):
        """
        Main function to run the BPE algorithm.
        """
        self.preprocess_and_count(all_sentences)

        for i in range(num_merges):
            pairs = self.get_pair_counts()
            if not pairs:
                break
            # Find the most common pair
            best_pair = max(pairs, key=pairs.get)
            self.merge_vocab(best_pair)
            # Optional progress information
            if (i + 1) % 500 == 0:
                print(f"Completed {i + 1} merges")

        print(f"Final vocabulary size: {self.num_words}")

        print(f"Total merges learned: {len(self.merge_order)}")

    def _word_tokenizer(self, word):
        word_tuple = tuple(list(word) + ["</w>"])

        for pair in self.merge_order:
            new_word = []
            i = 0
            while i < len(word_tuple):
                if (
                    i < len(word_tuple) - 1
                    and word_tuple[i] == pair[0]
                    and word_tuple[i + 1] == pair[1]
                ):
                    # when get match, Fuse them together
                    new_word.append(pair[0] + pair[1])
                    i += 2  # Skip the next character since we consumed it
                else:
                    new_word.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_word)

        return word_tuple

    def tokenize(
        self,
        sentence,
    ):
        sentence = sentence.lower()
        sentence = sentence.replace("\n", " ")
        sentence = re.sub(r'([.,!?;:"\'()])', r" \1 ", sentence)

        words = sentence.strip().split()

        tokens = []
        # 3. Process every word through the BPE tokenizer
        for word in words:
            bpe_tokens = self._word_tokenizer(word)
            tokens.extend(bpe_tokens)
        return tokens

    def numericalize(self, tokens):

        numerical_ids = [
            self.word2idx.get(token, self.word2idx["<UNK>"]) for token in tokens
        ]

        return numerical_ids


In [ ]:
tokenizer = LLaMABPETokenizer()
tokenizer.fit(training_texts, num_merges=5000)

In [ ]:
block_size = 128

all_sentences_idx = []
for sentence in training_texts:
    tokenized_idx = tokenizer.encode(sentence)
    all_sentences_idx.extend(tokenized_idx)

num_blocks = len(all_sentences_idx) // block_size
all_texts = [
    all_sentences_idx[i * block_size : (i + 1) * block_size] for i in range(num_blocks)
]


In [ ]:
class LanguageModelDataset(Dataset):
    def __init__(self, tokens):

        self.tokens = tokens

    def __len__(self):

        return len(self.tokens)

    def __getitem__(self, idx):
        src_tensor = torch.tensor(
            self.tokens[idx],
            dtype=torch.long,
        )

        return src_tensor

In [ ]:
# def collate_fn_for_unsupervised_learning(batch):

#     padded_batch = pad_sequence(batch, padding_value=0, batch_first=True)

#     return padded_batch


In [ ]:
dataset = LanguageModelDataset(all_texts)


dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    #  collate_fn=collate_fn_for_unsupervised_learning,
)


src_batch = next(iter(dataloader))

print("\nSource batch shape:", src_batch.shape)

print("\nSource batch:")

print(src_batch)

In [ ]:
class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len=5000):
        super().__init__()
        self.vocab_size = vocab_size
        self.emb_dim = d_model
        self.max_seq_len = max_seq_len
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, src):
        batch_size, seq_len = src.shape
        if seq_len > self.max_seq_len:
            raise ValueError(
                f"Sequence length ({seq_len}) exceeds max_seq_len ({self.max_seq_len})"
            )

        embedded = self.embedding(src)

        output_embedding = self.dropout(embedded)

        return output_embedding

In [ ]:
class RotaryPositionalEncodingConceptual(nn.Module):
    def __init__(self, base=10000):
        super().__init__()
        self.base = base

    def forward(self, src):
        # src: (batch_size, seq_len, emb_dim)

        batch_size, seq_len, emb_dim = src.shape

        if emb_dim % 2 != 0:
            raise ValueError("emb_dim must be even")

        rotated_src = torch.zeros_like(src)

        for token in range(seq_len):
            for i in range(0, emb_dim, 2):
                theta = self.base ** (-i / emb_dim)

                # Position-dependent rotation
                angle = token * theta

                cos = math.cos(angle)
                sin = math.sin(angle)

                # Rotation
                x1 = src[:, token, i]
                x2 = src[:, token, i + 1]

                rotated_src[:, token, i] = x1 * cos - x2 * sin

                rotated_src[:, token, i + 1] = x1 * sin + x2 * cos

        return rotated_src

In [ ]:
class RotaryEmbeddingAdjacent(nn.Module):
    def __init__(self, dim: int, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, emb_dim = src.shape
        if emb_dim % 2 != 0:
            raise ValueError("emb_dim must be even for positional encoding")

        pos = torch.arange(seq_len, device=src.device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", pos, self.inv_freq)

        emb = torch.repeat_interleave(freqs, 2, dim=-1)
        cos = emb.cos().unsqueeze(0).unsqueeze(0)
        sin = emb.sin().unsqueeze(0).unsqueeze(0)

        x1 = src[..., 0::2]
        x2 = src[..., 1::2]

        rotated_src = torch.stack((-x2, x1), dim=-1).flatten(-2)

        return (src * cos) + (rotated_src * sin)


class RotaryEmbeddingHalfSplit(nn.Module):
    def __init__(self, dim: int, base: float = 10000.0):
        super().__init__()
        self.dim = dim
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        batch_size, heads, seq_len, emb_dim = src.shape
        if emb_dim != self.dim:
            raise ValueError(f"emb_dim ({emb_dim}) must match init dim ({self.dim})")
        if emb_dim % 2 != 0:
            raise ValueError("emb_dim must be even for positional encoding")

        pos = torch.arange(seq_len, device=src.device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", pos, self.inv_freq)

        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos().unsqueeze(0).unsqueeze(0)
        sin = emb.sin().unsqueeze(0).unsqueeze(0)

        x1 = src[..., : emb_dim // 2]
        x2 = src[..., emb_dim // 2 :]

        rotated_src = torch.cat((-x2, x1), dim=-1)

        return (src * cos) + (rotated_src * sin)


In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        r_sqrt = torch.rsqrt(variance + self.eps)
        rms = x * r_sqrt * self.weight
        return rms


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, max_seq_len=5000):
        super().__init__()
        self.h = h
        self.d_k = d_k
        self.d_v = d_v
        self.attention_dropout = nn.Dropout(0.1)
        self.Wq = nn.Linear(d_model, h * d_k)
        self.Wk = nn.Linear(d_model, h * d_k)
        self.Wv = nn.Linear(d_model, h * d_v)
        self.rotationary_embedding = RotaryEmbeddingHalfSplit(dim=d_k)
        self.feedforward = nn.Linear(h * d_v, d_model)
        self.register_buffer(
            "causal_mask",
            torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool(),
        )

    def forward(self, src, mask=False, padd_mask=None):
        batch_size = src.size(0)
        seq_len = src.size(1)
        # Project and reshape
        Q = self.Wq(src).view(batch_size, seq_len, self.h, self.d_k).transpose(1, 2)

        K = self.Wk(src).view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
        V = self.Wv(src).view(batch_size, -1, self.h, self.d_v).transpose(1, 2)

        rotated_Q = self.rotationary_embedding(Q)
        rotated_K = self.rotationary_embedding(K)
        scores = torch.matmul(rotated_Q, rotated_K.transpose(-2, -1)) / math.sqrt(
            self.d_k
        )

        # Apply masks
        if mask:
            scores.masked_fill_(self.causal_mask[:seq_len, :seq_len], float("-inf"))

        if padd_mask is not None:
            scores.masked_fill_(padd_mask.unsqueeze(1).unsqueeze(2), float("-inf"))

        # Attention weights and output
        attention = F.softmax(scores, dim=-1)
        add_dropout = self.attention_dropout(attention)
        output = torch.matmul(add_dropout, V)

        # Combine heads and project
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.feedforward(output)

In [ ]:
class AttentionSublayer(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, dropout=0.1):
        super().__init__()
        self.multi_head_attention_sublayer = MultiHeadAttention(h, d_model, d_k, d_v)

        self.normalization = RMSNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, mask=False, padd_mask=None):
        normalized_src = self.normalization(src)
        attention_output = self.multi_head_attention_sublayer(
            src=normalized_src,
            mask=mask,
            padd_mask=padd_mask,
        )

        add_dropout = self.dropout(attention_output)
        add_residual = src + add_dropout

        return add_residual


In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d_model: int, d_ff: int = None):
        super().__init__()
        if d_ff is None:
            # LLaMA calculation for hidden dimension in FFN
            d_ff = int(2 * 4 * d_model / 3)
            d_ff = 256 * ((d_ff + 256 - 1) // 256)

        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # Gate projection
        self.w2 = nn.Linear(d_ff, d_model, bias=False)  # Down projection
        self.w3 = nn.Linear(d_model, d_ff, bias=False)  # Up projection

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.w1(src))
        up = self.w3(src)
        activated = gate * up

        down_proj = self.w2(activated)
        return down_proj

In [ ]:
class FeedForwardSublayer(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()

        self.normalization = RMSNorm(d_model)
        self.swiglu = SwiGLU(d_model, d_ff)

    def forward(self, src):
        norm_src = self.normalization(src)
        return src + self.swiglu(norm_src)

In [ ]:
class Decoder(nn.Module):
    def __init__(self, h, d_model, d_k, d_v, embedding, vocab_size, n_layers=6):
        super().__init__()
        self.embedding_layer = embedding
        self.classifier_dropout = nn.Dropout(0.1)
        self.normalization = RMSNorm(d_model)
        self.vocab_size = vocab_size
        self.layers = nn.ModuleList(
            [
                nn.ModuleDict(
                    {
                        "self_attention": AttentionSublayer(h, d_model, d_k, d_v),
                        "feed_forward": FeedForwardSublayer(d_model),
                    }
                )
                for layers in range(n_layers)
            ]
        )
        self.lm_head = nn.Linear(d_model, self.vocab_size, bias=False)

    def forward(
        self,
        data,
    ):
        trg_padd_mask = data == 0

        decoded = self.embedding_layer(data)
        for layer in self.layers:
            self_attention = layer["self_attention"](
                src=decoded,
                mask=True,
                padd_mask=trg_padd_mask,
            )

            feedforward_output = layer["feed_forward"](self_attention)
            decoded = feedforward_output
        decoded = self.normalization(decoded)
        output = self.lm_head(decoded)
        return output


In [ ]:
class LLaMA(nn.Module):
    def __init__(self, decoder, device):
        super().__init__()
        self.decoder = decoder
        self.device = device

    def forward(self, data):

        decoder_output = self.decoder(data)

        return decoder_output


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = len(tokenizer.word2idx)
D_MODEL = 4096
H = 32
D_V = D_MODEL // H
D_K = D_MODEL // H

n_layers = 32

embedding_layer = EmbeddingLayer(
    vocab_size=VOCAB_SIZE, d_model=D_MODEL, max_seq_len=5000
)

dec = Decoder(
    h=H,
    d_model=D_MODEL,
    d_k=D_K,
    d_v=D_V,
    embedding=embedding_layer,
    vocab_size=VOCAB_SIZE,
    n_layers=n_layers,
)
model = LLaMA(decoder=dec, device=device).to(device)


max_lr = 3.0e-4
beta_1 = 0.9
beta_2 = 0.95
epsilon = 1e-9
lambda_ = 0.5
clip = 1
label_smoothing = 0.1
N_EPOCHS = 10
train_steps = N_EPOCHS * len(dataloader)
warmup_steps = int(0.05 * train_steps)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=max_lr,
    betas=(beta_1, beta_2),
    eps=epsilon,
    weight_decay=0.1,
)


def lr_multiplier(step, warmup_steps=warmup_steps, total_steps=train_steps):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))


lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_multiplier)

criterion = nn.CrossEntropyLoss(
    ignore_index=0,
    label_smoothing=label_smoothing,
)

print(f"Model initialized successfully. Device: {device}")

In [ ]:
tensor_2d = torch.tensor([[1, 2, 3, 4, 5], [6, 7, 8, 9, 0]])
tensor_2d.shape
inputs = tensor_2d[:, :-1]
targets = tensor_2d[:, 1:]
print(inputs)
print(targets)

In [ ]:
def training(model, dataloader, optimizer, criterion, clip):

    model.train()
    epoch_loss = 0

    for data in dataloader:
        data = data.to(device)
        inputs = data[:, :-1]
        targets = data[:, 1:]
        optimizer.zero_grad()
        output = model(inputs)

        flat_trg = targets.reshape(-1)
        flat_output = output.reshape(-1, VOCAB_SIZE)

        loss = criterion(flat_output, flat_trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()
        lr_scheduler.step()
        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)


for epoch in range(N_EPOCHS):
    start_time = time.time()

    train_loss = training(model, dataloader, optimizer, criterion, clip)
    end_time = time.time()

    print(f"Epoch: {epoch + 1:02} | Time: {end_time - start_time:.2f}s")
    print(
        f"\tTrain Loss: {train_loss:.3f} | Jarring Perplexity: {math.exp(train_loss):.3f}"
    )


In [ ]:
def zero_shot_test(sentence, device, tokenizer, model, max_new_tokens=100):
    model.eval()
    tokens = tokenizer.encode(sentence)

    token_tensor = torch.tensor(
        tokens,
        dtype=torch.long,
    )
    token_tensor = token_tensor.unsqueeze(0).to(device)
    token_indexes = []
    for _ in range(max_new_tokens):
        with torch.no_grad():
            model_output = model(token_tensor)
            # model ouput = batchsize,seq_len,vocab
        pred_token = model_output[0, -1, :].argmax().item()
        token_indexes.append(pred_token)
        token_tensor = (
            torch.tensor(
                tokens + token_indexes,
                dtype=torch.long,
            )
            .unsqueeze(0)
            .to(device)
        )
        if pred_token == tokenizer.word2idx["</s>"]:
            break
    trg_tokens = [tokenizer.idx2word[i] for i in token_indexes]
    return trg_tokens


example_input = "tell me a story about rats"
sample_output = zero_shot_test(example_input, device, tokenizer, model)
print("input: ", example_input)
print("output: ", sample_output)